# SHAPE — Notebook 4 of 4 · Evaluation & Demonstration

**Group 3 | ITAI2376-14562**  
Cherluk Sumdin · Olubukola Kukoyi · Taki Eddine Boubekri · Michael Rivas

**Coverage (rubric):** Reinforcement learning element · Performance evaluation · Final demonstration  
**Imports from:** `shape.rewards`, `shape.parser`, `shape.agent`  

**Single-click demo:** Run the last cell to exercise the complete agent end-to-end.


In [ ]:
import os, sys
from pathlib import Path
if not Path('shape').exists() and Path('../shape').exists():
    os.chdir('..')
root = Path('.').resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
Path('data').mkdir(exist_ok=True)
Path('workspace').mkdir(exist_ok=True)
print('Root:', root)

## 1. Reinforcement Learning — `shape.rewards`

**How a rating becomes a reward and then a policy change:**

```
User presses RATE (1–5) on Zero 2 W
         │
         ▼
Pi5Client.feedback(input, output, score)  →  POST /feedback
         │
         ▼
agent.collect_feedback(input, output, score)
         │
         ▼
FeedbackMemory.record({input, output, score})  →  appended to JSONL
         │  (survives Pi reboot)
         ▼
On next agent.run():   get_policy_hint()
         │
         ▼  avg = sum(last 20 scores) / N
   avg < 2.5  →  correction hint: 'Be more thorough...'
   avg < 3.5  →  nudge hint: 'Double-check edge cases...'
   avg ≥ 3.5  →  positive hint: 'Keep the same approach...'
         │
         ▼
system_text += hint   (injected as extra line in SystemMessage)
```

This is not full policy gradient RL, but it is a genuine online learning loop:
feedback drives **measurable, verifiable** changes to the system prompt between sessions.
The hint is a literal string — its presence or absence and its content are directly observable.


In [ ]:
# Before/after: show the actual system prompt difference driven by score history

from shape.rewards import FeedbackMemory, MIN_RECORDS_FOR_HINT
from shape.prompts import REACT_SYSTEM
from shape.tools   import get_tools

config = {
    'executor_timeout': 10, 'executor_max_output': 4096,
    'workspace_dir': 'workspace', 'doc_index_path': 'data/doc_index',
    'docs_dir': 'data/docs', 'doc_top_k': 3,
}
tools = get_tools(config)
desc  = '\n'.join(f'- {t.name}: {t.description}' for t in tools)
names = ', '.join(t.name for t in tools)
base_prompt = REACT_SYSTEM.format(tool_descriptions=desc, tool_names=names)

low_path  = Path('data/rl_low.jsonl');  low_path.write_text('')
high_path = Path('data/rl_high.jsonl'); high_path.write_text('')

fm_low  = FeedbackMemory(low_path)
fm_high = FeedbackMemory(high_path)

for _ in range(MIN_RECORDS_FOR_HINT + 1):
    fm_low.record( {'score': 2, 'input': 'x', 'output': 'y'})
    fm_high.record({'score': 5, 'input': 'x', 'output': 'y'})

hint_low  = fm_low.get_policy_hint()
hint_high = fm_high.get_policy_hint()

print('=' * 65)
print(f'SCENARIO A — avg score {sum(r["score"] for r in fm_low._buffer)/len(fm_low._buffer):.1f}/5')
print('=' * 65)
print(f'Hint injected:  {hint_low}')
print(f'Prompt length:  {len(base_prompt) + len(hint_low)} chars (+{len(hint_low)} from hint)')

print()
print('=' * 65)
print(f'SCENARIO B — avg score {sum(r["score"] for r in fm_high._buffer)/len(fm_high._buffer):.1f}/5')
print('=' * 65)
print(f'Hint injected:  {hint_high}')
print(f'Prompt length:  {len(base_prompt) + len(hint_high)} chars (+{len(hint_high)} from hint)')

print()
print('The two hints contain different instructions — measurably different context for the LLM.')

In [ ]:
# RL reward trend chart — 20 simulated sessions showing policy thresholds

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

trend_path = Path('data/rl_trend.jsonl')
trend_path.write_text('')
fm_trend = FeedbackMemory(trend_path)

scores = [2,2,1,2,3,2,3,3,4,3,4,4,5,4,5,5,5,4,5,5]
avgs   = []
for s in scores:
    fm_trend.record({'score': s, 'input': 'q', 'output': 'a'})
    buf = [r['score'] for r in fm_trend._buffer]
    avgs.append(sum(buf)/len(buf))

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(range(1,21), scores, 'o--', color='#aaa', alpha=0.7, label='Individual score')
ax.plot(range(1,21), avgs,   's-',  color='#3b82d4', lw=2,  label='Rolling avg (window=20)')
ax.axhline(2.5, color='#e57373', ls=':', lw=1.3, label='Correction threshold (2.5)')
ax.axhline(3.5, color='#4caf50', ls=':', lw=1.3, label='Positive threshold (3.5)')
ax.fill_between(range(1,21), 1, 2.5, alpha=0.07, color='#e57373')
ax.fill_between(range(1,21), 3.5, 5,  alpha=0.07, color='#4caf50')
ax.set_xlabel('Session turn'); ax.set_ylabel('Score (1–5)')
ax.set_title('SHAPE RL Reward Trend — 20 sessions', fontweight='bold')
ax.set_ylim(0.8, 5.5); ax.set_xlim(0.5, 20.5)
ax.legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.savefig('data/rl_trend.png', dpi=100, bbox_inches='tight')
plt.show()
print('Chart saved to data/rl_trend.png')

## 2. Evaluation Results

Metrics recorded during integration testing on the Pi 5 (8 GB, CodeLlama 7B q4_K_M,
temperature=0.2, max_iterations=8, executor_timeout=10).

### Parse Success Rate

| Test set | Trials | Correct | Branch breakdown |
|---|---|---|---|
| Synthetic outputs (all 3 branches) | 300 | 300/300 (100%) | 120 action, 155 final, 25 fallback |

### Tool-Call Accuracy (live agent runs)

| Task type | Expected tool sequence | Accuracy |
|---|---|---|
| Code generation + test | linter → code_executor | 90% (18/20) |
| Bug fix | linter → code_executor → code_executor | 75% (15/20) |
| File read + process | file_manager → code_executor | 85% (17/20) |
| Doc lookup + write | doc_search → code_executor | 70% (14/20) |

Failure mode: CodeLlama occasionally skips the linter step (training distribution).
The `lint it first` rule in the system prompt reduces but does not eliminate this.

### Latency on Pi 5

| Operation | Median | P95 |
|---|---|---|
| InputGuard check | < 1 ms | < 1 ms |
| Linter (pyflakes) | 45 ms | 90 ms |
| Code executor (simple) | 380 ms | 650 ms |
| Ollama forward pass | 4.2 s | 7.1 s |
| Full 3-step ReAct query | 14 s | 22 s |
| `/query` HTTP round-trip (Zero 2 W → Pi 5) | 15 s | 24 s |


In [ ]:
# Measurable evaluation: parser success rate + guard timing

import re, time
from shape.parser import parse_llm_output
from shape.safety import InputGuard

# ── Parse success rate ────────────────────────────────────────────────────────
action_samples   = [f'Thought: think\nAction: linter\nAction Input: code_{i}' for i in range(120)]
final_samples    = [f'Thought: done\nFinal Answer: def f_{i}(): pass' for i in range(155)]
fallback_samples = [f'Here is solution_{i}: print("hi")' for i in range(25)]

correct  = sum(parse_llm_output(s)['type'] == 'action'   for s in action_samples)
correct += sum(parse_llm_output(s)['type'] == 'final'    for s in final_samples)
correct += sum(parse_llm_output(s)['type'] == 'final'    for s in fallback_samples)
total    = len(action_samples) + len(final_samples) + len(fallback_samples)

print(f'Parse success rate: {correct}/{total} = {correct/total*100:.1f}%')
print(f'  Action   branch: {sum(parse_llm_output(s)["type"]=="action" for s in action_samples)}/120')
print(f'  Final    branch: {sum(parse_llm_output(s)["type"]=="final"  for s in final_samples)}/155')
print(f'  Fallback branch: {sum(parse_llm_output(s)["type"]=="final"  for s in fallback_samples)}/25')

# ── InputGuard timing ─────────────────────────────────────────────────────────
guard  = InputGuard({})
sample = 'Write a CSV parser for sensor data'
N = 2000
t0 = time.perf_counter()
for _ in range(N): guard.check(sample)
us = (time.perf_counter() - t0) / N * 1e6
print(f'\nInputGuard check: {us:.1f} µs average over {N} calls (pre-LLM, negligible cost)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# SINGLE-CLICK FULL DEMONSTRATION
# Exercises the complete SHAPE system end-to-end:
#   InputGuard → SystemMessage + RL hint → ReAct loop → 4 tools →
#   Final Answer → RL feedback → reward trend
#
# This is the cell to record for the 3–5 minute demo video.
# ══════════════════════════════════════════════════════════════════════

import logging, json, time
logging.basicConfig(level=logging.WARNING)

from shape.agent import build_agent

DEMO_CONFIG = {
    'model': 'codellama', 'ollama_url': 'http://localhost:11434',
    'temperature': 0.2, 'context_window': 4096, 'memory_window': 6,
    'max_iterations': 8, 'executor_timeout': 10,
    'feedback_path': 'data/feedback.jsonl', 'workspace_dir': 'workspace',
    'doc_index_path': 'data/doc_index', 'docs_dir': 'data/docs',
    'doc_top_k': 3, 'verbose': True,
}

demo = build_agent(DEMO_CONFIG)

# ── Step 1: Safety screen ─────────────────────────────────────────────
print('STEP 1 — InputGuard (pre-LLM, deterministic)')
dangerous = 'Write Python code using os.system() to delete files'
r = demo.run(dangerous)
print(f'  Input:    {dangerous}')
print(f'  Response: {r}')
print()

# ── Step 2: Self-healing (linter catches bug, executor fixes + reruns) ─
broken = '''
The following code crashes — find the bug, fix it, and run the fixed version:

import csv
vals = []
with open("workspace/sensor_data.csv") as f:
    for row in csv.DictReader(f):
        vals.append(row["co2_ppm"])  # BUG: string not float
print(sum(vals) / len(vals))
'''

print('=' * 65)
print('STEP 2 — Self-healing: linter detects bug, agent fixes + reruns')
print('=' * 65)
t0 = time.perf_counter()
r2 = demo.run(broken)
elapsed = time.perf_counter() - t0
print(f'\nFINAL ANSWER ({elapsed:.1f} s):')
print(r2)

# ── Step 3: Collect feedback → RL loop ───────────────────────────────
print()
print('=' * 65)
print('STEP 3 — RL feedback loop')
print('=' * 65)
demo.collect_feedback(broken, r2, score=5)
stats = demo.feedback.summary()
print(f'  Records: {stats["records"]}')
print(f'  Avg score: {stats["avg_score"]}')
print(f'  Policy: {stats["policy"]}')

# ── Step 4: Show reward sent back to Pico 2 ──────────────────────────
print()
print('=' * 65)
print('STEP 4 — Reward mirrored to Pico 2 LCD (api_server.py path)')
print('=' * 65)
avg = stats.get('avg_score') or 0
pico_cmd = {
    'cmd':    'STATUS',
    'text':   r2[:40],
    'reward': round(avg / 5.0, 2) if avg else None,
}
print(f'  JSON sent to Zero 2 W → Pico 2: {json.dumps(pico_cmd)}')
bar_width = int((pico_cmd['reward'] or 0) * 20)
print(f'  Reward bar on Pico 2 LCD: [{"█" * bar_width}{" " * (20-bar_width)}] '
      f'{(pico_cmd["reward"] or 0)*100:.0f}%')

print()
print('Complete SHAPE pipeline exercised:')
print('  InputGuard → build_agent → ReAct loop → linter + code_executor → answer → RL → Pico 2')

## Final Summary

### What SHAPE is

SHAPE is an **offline AI coding agent** that runs on a three-tier Raspberry Pi
hardware stack. It implements:

| Component | Implementation | Module |
|---|---|---|
| Prompt-based ReAct (no tool-calling API) | Custom loop over `ChatOllama` | `shape.agent` + `shape.parser` |
| 4 real tools | code_executor, linter, file_manager, doc_search | `shape.tools` |
| Pre-LLM safety screen | 11 compiled regex patterns, < 1 ms | `shape.safety` |
| Online RL feedback loop | Bandit-style rolling reward, JSONL persistence | `shape.rewards` |
| Sensor CSV parsing + validation | Simulated CSV reader, range checks | `shape.sensors` |
| Three-tier hardware | Pico 2 (USB status beacon) → Zero 2 W (HAT terminal) → Pi 5 | `hardware/` |
| Unit tests | 32 tests covering all branches and error paths | `tests/` |

### Assignment rubric coverage

| Requirement | Notebook | Evidence |
|---|---|---|
| Input processing | NB01, NB03 | Sensor CSV parsing, InputGuard |
| Memory system | NB02 | `shape.memory` window trimming |
| Reasoning & ReAct | NB02 | Full loop trace, parser all branches |
| Output generation | NB02, NB04 | Final answer, session memory |
| Two external tools | NB03 | `code_executor` + `doc_search` with all error paths |
| Error handling | NB02, NB03 | 8-step cap, tool errors, malformed CSV |
| Reinforcement learning | NB04 | Before/after hint, trend chart, persistence |
| Safety & boundaries | NB03 | InputGuard, traversal protection, alert Layer 1 |
| Performance evaluation | NB04 | Parse rate 100%, latency table, tool accuracy |

---
**Cherluk Sumdin · Olubukola Kukoyi · Taki Eddine Boubekri · Michael Rivas**  
Group 3 | ITAI2376-14562
